# Jumper su Kaggle — training della scalinata a balzi (`jumper.stairs`)

**Prima di tutto (⚙ Settings, barra a destra):** Accelerator **GPU T4 x2** (mai P100), Internet **On**, e in *Input*: (1) il Dataset `jumper-stairs-task` con il codice della task, (2) per continuare un training, la versione precedente di questo notebook (Add Input → Your Work → Notebooks).

**Come si usa.** Si modifica solo la cella 1, poi **Save Version → Save & Run All (Commit)**: il notebook gira in background a browser chiuso, fino a 12 h, e alla fine `/kaggle/working/jumper/` (backup ZIP, curve, video, export) resta come output della versione. *Run All* interattivo va bene per provare, ma l'output si perde se non lo scarichi dal pannello Output.

**Cosa fa.** Setup identico al notebook ufficiale (clone al commit bloccato, venv isolato, torch cu126, EGL, check CUDA/Warp) + copia del codice della task dal Dataset → punto di partenza (ripristino del backup, oppure warm start dall'actor di `ref_free_jump`) → mini-run di prova → training lungo, una task per GPU, fermato a orologio dopo `TRAIN_HOURS` → curve, metriche della scalinata, replay MP4, export ONNX (solo la task cieca), backup ZIP.

**Cosa guardare nel log del training (cella 4, una riga al minuto):** `level` (riga del curriculum, 0–9 → alzata 2–15 cm), `steps_by_flight` (gradini saliti saltando per episodio), e nella cella 5 `crawl_ratio` (deve restare vicino a 0: se sale, il granchio striscia).


In [ ]:
# @title 1. Impostazioni (modifica solo questa cella)
import json
import math
import os
import re
import shutil
import signal
import subprocess
import sys
import threading
import time
import uuid
import venv
import zipfile
from collections import deque
from datetime import datetime, timezone
from pathlib import Path

# ── Cosa allenare ────────────────────────────────────────────────────────────
TASK = "jumper.stairs"                 # GPU 0: "jumper.stairs" (A, cieco, esportabile)
SECOND_TASK = "jumper.stairs_sighted"  # GPU 1: "jumper.stairs_sighted" (B, vede il terreno) oppure "" per nessuno
NUM_ENVS = 2048                        # per processo; su T4 16 GB sotto i 12 000 MiB di picco c'è margine
WORKFLOW_MODE = "New training"         # "New training" oppure "Continue training"

# ── Tempo ────────────────────────────────────────────────────────────────────
TRAIN_HOURS = 10.0      # training; poi ~1 h per backup, curve, replay, export. Kaggle taglia a 12 h.
SMOKE_ITERATIONS = 5    # mini-run di prova prima di quello lungo (0 per saltarlo)

# ── Warm start (solo New training) ───────────────────────────────────────────
WARM_START = True
WARM_SOURCE = "tasks/jumper/ref_free_jump/out/example/model_5999.pt"

# ── Sorgenti ─────────────────────────────────────────────────────────────────
REPOSITORY_URL = "https://github.com/KingKongRobotics/jumper.git"
SOURCE_REVISION = "b2e58669848a3ba4f209597cd6bebdfe9d0533a4"   # bloccato: la ripresa lo confronta
PATCH_DIR = "/kaggle/input/jumper-stairs-task"   # il Dataset con il codice della task (zip o cartella)
BACKUP_GLOB = "/kaggle/input/*/backups/*.zip"    # Continue: i backup delle versioni precedenti (Add Input)

# ── Replay ───────────────────────────────────────────────────────────────────
REPLAY_SECONDS = 12.0
REPLAY_ROW = ""         # "" = la riga media raggiunta dal curriculum; oppure 0..9
REPLAY_WIDTH, REPLAY_HEIGHT, REPLAY_FPS = 960, 540, 30.0

# ── Derivati ─────────────────────────────────────────────────────────────────
ON_KAGGLE = Path("/kaggle").is_dir()
WORK = Path("/kaggle/temp") if ON_KAGGLE else Path("/tmp/jumper-work")     # effimero
OUT = Path("/kaggle/working/jumper") if ON_KAGGLE else Path.cwd() / "jumper-out"  # persistente (output)
REPO = WORK / "jumper"
PYTHON = REPO / ".venv/bin/python"
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
SESSION = OUT / f"session-{STAMP}"
BACKUPS = OUT / "backups"
TASKS = [t for t in (TASK, SECOND_TASK) if t]
for directory in (WORK, OUT, SESSION, BACKUPS):
    directory.mkdir(parents=True, exist_ok=True)
if WORKFLOW_MODE not in ("New training", "Continue training"):
    raise ValueError("WORKFLOW_MODE: New training oppure Continue training")
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError(f"Jumper vuole Python 3.10-3.13, qui c'è {sys.version}")
for t in TASKS:
    if not re.fullmatch(r"jumper\.[a-z_]+", t):
        raise ValueError(f"task non valida: {t}")
print(f"{WORKFLOW_MODE}: {TASKS} x {NUM_ENVS} env, {TRAIN_HOURS} h di training. Output in {OUT}")


In [ ]:
# @title 2. Setup: GPU, clone, venv, torch, EGL, codice della task  (~6-10 min)
def run(command, *, cwd=None, check=True, env=None, timeout_s=None, tail_lines=80):
    """Esegue e mostra l'output in diretta. Se scade `timeout_s`, SIGINT a tutto
    il gruppo (training e helper salvano e chiudono) e si rientra con quel codice."""
    print("+", " ".join(str(p) for p in command), flush=True)
    command = [str(p) for p in command]
    tail = deque(maxlen=tail_lines)
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    timer = None
    if timeout_s:
        timer = threading.Timer(timeout_s, lambda: os.killpg(process.pid, signal.SIGINT))
        timer.daemon = True
        timer.start()
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line[-4096:])
        code = process.wait()
        if code and check:
            raise subprocess.CalledProcessError(code, command, output="".join(tail))
        return subprocess.CompletedProcess(command, code, stdout="".join(tail))
    except BaseException:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGINT)
                process.wait(timeout=30)
            except (ProcessLookupError, subprocess.TimeoutExpired):
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        raise
    finally:
        if timer:
            timer.cancel()
        process.stdout.close()


# ── GPU ──
run(["nvidia-smi", "-L"])
rows = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total", "--format=csv,noheader"], text=True,
).strip().splitlines()
if not rows:
    raise RuntimeError("Nessuna GPU: Settings → Accelerator → GPU T4 x2")
gpus = [[p.strip() for p in r.split(",")] for r in rows]
capability = float(gpus[0][2])
if capability < 7.5:
    raise RuntimeError(f"GPU {gpus[0][1]} (compute capability {capability}) non supportata: scegli T4 x2, non P100")
GPU_NAME = gpus[0][1]
if SECOND_TASK and len(gpus) < 2:
    raise RuntimeError("SECOND_TASK vuole due GPU: scegli T4 x2 oppure metti SECOND_TASK = ''")
print("GPU:", [g[1] for g in gpus])

# ── Clone al commit bloccato ──
if not REPO.exists():
    run(["git", "clone", REPOSITORY_URL, REPO])
run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO)
run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()

# ── Il codice della task, dal Dataset, sopra il clone ──
patch = Path(PATCH_DIR)
if not patch.exists():
    raise FileNotFoundError(f"PATCH_DIR {patch} non esiste: Add Input → il dataset con il codice della task")
zips = sorted(patch.rglob("*.zip"))
if zips:
    with zipfile.ZipFile(zips[-1]) as z:
        names = z.namelist()
        if any(n.startswith("/") or ".." in Path(n).parts for n in names):
            raise ValueError("zip con percorsi non sicuri")
        z.extractall(REPO)
    print("Applicato", zips[-1], f"({len(names)} file)")
else:
    copied = 0
    for src in patch.rglob("*"):
        if src.is_file():
            rel = src.relative_to(patch)
            # la cartella del dataset può avere un livello in più (nome del dataset)
            parts = rel.parts
            if parts and parts[0] not in ("tasks", "tools", "notebooks", "docs"):
                rel = Path(*parts[1:])
            dst = REPO / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            copied += 1
    print(f"Copiati {copied} file da {patch}")
for required in ("tasks/jumper/stairs/env_cfg.py", "tools/warm_start.py"):
    if not (REPO / required).is_file():
        raise FileNotFoundError(f"manca {required}: il Dataset non contiene il codice della task")

# ── Continue: quale backup per ogni task, e le sue versioni dei pacchetti ──
import glob

def backup_manifest(archive):
    with zipfile.ZipFile(archive) as z:
        info = z.getinfo("run/colab-run.json")
        if info.file_size > 1024**2:
            raise ValueError("manifest troppo grande")
        return json.loads(z.read(info))

BACKUP_FOR = {}
CORE_VERSIONS = {}
if WORKFLOW_MODE == "Continue training":
    candidates = []
    for archive in sorted(glob.glob(BACKUP_GLOB)):
        try:
            m = backup_manifest(archive)
        except Exception as exc:  # noqa: BLE001
            print("salto", archive, "->", exc)
            continue
        cfg = m.get("config", {})
        if m.get("schema") == 1 and m.get("checkpoints") and cfg.get("task") in TASKS:
            candidates.append((Path(archive).stat().st_mtime, archive, cfg))
    for t in TASKS:
        mine = [c for c in candidates if c[2]["task"] == t]
        if not mine:
            raise FileNotFoundError(f"nessun backup di {t} in {BACKUP_GLOB}: Add Input → la versione precedente del notebook")
        _, archive, cfg = max(mine)
        BACKUP_FOR[t] = archive
        if cfg["revision"] != SOURCE_REVISION or cfg["num_envs"] != NUM_ENVS:
            raise ValueError(f"il backup di {t} è del commit {cfg['revision'][:9]} a {cfg['num_envs']} env: "
                             f"SOURCE_REVISION e NUM_ENVS devono coincidere")
        if CORE_VERSIONS and CORE_VERSIONS != cfg["runtime_versions"]:
            raise ValueError("i due backup hanno versioni dei pacchetti diverse")
        CORE_VERSIONS = dict(cfg["runtime_versions"])
        print(f"Continue {t} da {archive}")
    if CORE_VERSIONS.get("torch", "2.9.1").split("+")[0] != "2.9.1":
        raise ValueError("questo notebook supporta torch 2.9.1")

# ── venv isolato ──
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(REPO / ".venv")
run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "--upgrade", "pip", "-q"])
try:
    run(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "ffmpeg"], check=False)
except Exception as exc:  # noqa: BLE001
    print("apt-get non riuscito (continuo):", exc)
TORCH_INDEX = "cu126" if capability < 8.0 else "cu128"
run([PYTHON, "-m", "pip", "install", "-q", "torch==2.9.1", "torchvision==0.24.1",
     "--index-url", f"https://download.pytorch.org/whl/{TORCH_INDEX}"])
run([PYTHON, "-m", "pip", "install", "-q", "-e", str(REPO)])
pins = [f"{k}=={v}" for k, v in sorted(CORE_VERSIONS.items()) if k != "torch"]
if pins:
    run([PYTHON, "-m", "pip", "install", "-q", *pins])   # le versioni esatte del backup, come il notebook ufficiale
run([PYTHON, "-m", "pip", "install", "-q", "matplotlib"])

# ── EGL NVIDIA (replay MP4), dal notebook ufficiale ──
os.environ.update(MUJOCO_GL="egl", PYTHONUNBUFFERED="1", MPLBACKEND="Agg", PYOPENGL_PLATFORM="egl")
lib = next((Path(c).resolve() for c in (
    "/usr/lib64-nvidia/libEGL_nvidia.so.0", "/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0",
    "/usr/lib/x86_64-linux-gnu/nvidia/current/libEGL_nvidia.so.0", "/usr/lib/nvidia/libEGL_nvidia.so.0",
) if Path(c).is_file()), None)
if lib is None:
    raise RuntimeError("libEGL_nvidia non trovata")
(WORK / "nvidia-egl.json").write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": str(lib)}}))
os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = str(WORK / "nvidia-egl.json")
os.environ["LD_LIBRARY_PATH"] = ":".join([str(lib.parent), *[e for e in os.environ.get("LD_LIBRARY_PATH", "").split(":") if e]])

run([PYTHON, "tools/colab.py", "check", "--repo", REPO, "--out", SESSION / "runtime.json"], cwd=REPO)
runtime = json.loads((SESSION / "runtime.json").read_text())
for pkg, want in CORE_VERSIONS.items():
    if runtime["versions"].get(pkg) != want:
        raise RuntimeError(f"versione di {pkg}: il backup vuole {want}, installata {runtime['versions'].get(pkg)}")
listing = run([PYTHON, "scripts/train.py", "--list"], cwd=REPO).stdout
for t in TASKS:
    if t not in listing:
        raise RuntimeError(f"{t} non compare in --list: manca la riga di import in tasks/__init__.py del Dataset")
run([PYTHON, "scripts/train.py", "--task", TASK, "--dry-run"], cwd=REPO)
print("\nSETUP OK —", GPU_NAME, "| commit", COMMIT[:9], "| versioni", runtime["versions"])


In [ ]:
# @title 3. Punto di partenza: ripristino del backup (Continue) oppure warm start (New)
START_FROM = {}   # task -> checkpoint da cui `tools/colab.py train --resume` riparte

if WORKFLOW_MODE == "Continue training":
    for t in TASKS:
        dest = WORK / f"restored-{t}-{uuid.uuid4().hex[:8]}"
        info = SESSION / f"restore-{t}.json"
        run([PYTHON, "tools/colab.py", "restore", "--archive", BACKUP_FOR[t], "--destination", dest, "--out", info], cwd=REPO)
        START_FROM[t] = json.loads(info.read_text())["checkpoint"]
        print(t, "riparte da", START_FROM[t])
elif WARM_START:
    # `tools/warm_start.py` scrive model_0.pt con l'actor del salto e critic/optimizer
    # freschi; `tools/colab.py train --resume` vuole accanto il suo manifest con la
    # configurazione e lo sha del file, e questo helper lo scrive con le funzioni di
    # colab.py stesso, così i campi di compatibilità sono esattamente quelli.
    manifest_helper = r"""
import importlib.util, json, sys, uuid
from pathlib import Path
repo, task, model, num_envs, ckpt = Path(sys.argv[1]), sys.argv[2], sys.argv[3], int(sys.argv[4]), Path(sys.argv[5])
spec = importlib.util.spec_from_file_location("colab", repo / "tools/colab.py")
colab = importlib.util.module_from_spec(spec); spec.loader.exec_module(colab)
cfg = colab.make_config(repo, task=task, model=model, num_envs=num_envs)
manifest = {"schema": colab.SCHEMA, "operation": "warm-" + uuid.uuid4().hex, "config": cfg,
            "command": ["tools/warm_start.py"], "status": "finished",
            "checkpoints": {ckpt.name: colab.sha256(ckpt)}, "checkpoint": str(ckpt),
            "resume_from": None, "run_directory": str(ckpt.parent)}
colab.atomic_json(ckpt.parent / colab.MANIFEST, manifest)
colab.validate_resume(ckpt, cfg)
print("manifest ok:", ckpt.parent / colab.MANIFEST)
"""
    for t in TASKS:
        out = REPO / "logs" / "warm" / t / "model_0.pt"
        run([PYTHON, "tools/warm_start.py", "--task", t, "--from", WARM_SOURCE, "--out", out,
             "--num-envs", "4", "--device", "cpu"], cwd=REPO)
        run([PYTHON, "-c", manifest_helper, REPO, t, "jumper", str(NUM_ENVS), out], cwd=REPO)
        START_FROM[t] = str(out)
else:
    print("New training da zero (WARM_START = False)")
print("Partenze:", START_FROM or "da zero")


In [ ]:
# @title 4. Mini-run di prova, poi il training lungo (una task per GPU, fermato a orologio)
def train_env(gpu_index):
    """Ogni processo vede una sola GPU, come `cuda:0`: colab.py allena sempre su cuda:0."""
    e = os.environ.copy()
    e["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
    return e


def train_command(task, iterations, manifest, num_envs, resume=None):
    cmd = [str(PYTHON), "tools/colab.py", "train", "--repo", str(REPO), "--task", task, "--model", "jumper",
           "--num-envs", str(num_envs), "--iterations", str(iterations), "--manifest", str(manifest)]
    if resume:
        cmd += ["--resume", str(resume)]
    return cmd


# ── Mini-run: se la scena non si costruisce (nconmax, un termine che non si risolve,
# un checkpoint che non si carica) deve fallire qui, in 3 minuti, non dopo 10 ore. ──
if SMOKE_ITERATIONS > 0:
    smoke_manifest = SESSION / f"smoke-{TASK}.json"
    # Senza --resume: il manifest del warm start è scritto per NUM_ENVS e colab.py
    # rifiuta un altro conteggio. Il caricamento del checkpoint si vede nei primi
    # minuti del run lungo, nel suo log.
    run(train_command(TASK, SMOKE_ITERATIONS, smoke_manifest, min(NUM_ENVS, 256)),
        cwd=REPO, env=train_env(0), timeout_s=900)
    smoke = json.loads(smoke_manifest.read_text())
    if smoke["status"] != "finished":
        raise RuntimeError(f"il mini-run non è finito: {smoke.get('status')} {smoke.get('error')}")
    print("Mini-run OK:", smoke["run_directory"])

# ── Il training lungo ──
MANIFESTS = {t: SESSION / f"training-{t}.json" for t in TASKS}
procs = {}
t0 = time.time()
for gpu, t in enumerate(TASKS):
    cmd = train_command(t, 1_000_000, MANIFESTS[t], NUM_ENVS, START_FROM.get(t))
    print("+", " ".join(cmd), f"(GPU {gpu})", flush=True)
    log = (SESSION / f"train-{t}.log").open("w")
    procs[t] = (subprocess.Popen(cmd, cwd=REPO, env=train_env(gpu), stdout=log, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1, start_new_session=True), log)

deadline = t0 + TRAIN_HOURS * 3600
stopped = False
PROGRESS = re.compile(r"Learning iteration (\d+)/|Mean reward:\s*([-0-9.]+)|Curriculum/stairs/level:\s*([-0-9.]+)|steps_by_flight:\s*([-0-9.]+)")
try:
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(60)
        now = time.time()
        status = []
        for t, (p, log) in procs.items():
            text = (SESSION / f"train-{t}.log").read_text(errors="replace")[-20000:]
            it = PROGRESS.findall(text)
            last_it = [m[0] for m in it if m[0]]
            last_rew = [m[1] for m in it if m[1]]
            last_lvl = [m[2] for m in it if m[2]]
            last_jmp = [m[3] for m in it if m[3]]
            status.append(f"{t}: it {last_it[-1] if last_it else '-'} reward {last_rew[-1] if last_rew else '-'}"
                          f" level {last_lvl[-1] if last_lvl else '-'} steps_by_flight {last_jmp[-1] if last_jmp else '-'}"
                          f"{'' if p.poll() is None else f' [uscito {p.returncode}]'}")
        print(f"[{(now - t0) / 3600:5.2f} h] " + " | ".join(status), flush=True)
        if now >= deadline and not stopped:
            print(">>> budget di tempo esaurito: SIGINT (il checkpoint più recente è già su disco)", flush=True)
            for p, _ in procs.values():
                if p.poll() is None:
                    os.killpg(p.pid, signal.SIGINT)
            stopped = True
        if stopped and now >= deadline + 300:
            for p, _ in procs.values():
                if p.poll() is None:
                    os.killpg(p.pid, signal.SIGKILL)
finally:
    for p, log in procs.values():
        try:
            p.wait(timeout=120)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL)
        log.close()

TRAINING = {}
RUN_DIR = {}
CHECKPOINT = {}
for t in TASKS:
    m = json.loads(MANIFESTS[t].read_text())
    print(f"{t}: stato {m['status']}, run {m.get('run_directory')}, checkpoint {m.get('checkpoint')}")
    if m["status"] not in ("finished", "interrupted") or not m.get("checkpoint"):
        print((SESSION / f"train-{t}.log").read_text(errors="replace")[-3000:])
        raise RuntimeError(f"{t}: training {m['status']}: {m.get('error')}")
    TRAINING[t] = m
    RUN_DIR[t] = Path(m["run_directory"])
    CHECKPOINT[t] = Path(m["checkpoint"])
print(f"Training chiuso dopo {(time.time() - t0) / 3600:.2f} h")


In [ ]:
# @title 5. Curve, metriche della scalinata, replay MP4, export ONNX, backup ZIP
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, Video, display

ARTIFACTS = {}
KEY_TAGS = (
    "Metrics/stairs/steps_by_flight", "Metrics/stairs/steps_crawled", "Metrics/stairs/crawl_ratio",
    "Metrics/stairs/jumps", "Metrics/stairs/apex_rise", "Metrics/stairs/flight_s", "Metrics/stairs/max_step",
    "Metrics/stairs/reached_top", "Curriculum/stairs/level", "Curriculum/stairs/riser_m",
    "Episode_Termination/fell_over", "Episode_Termination/fell_off", "Episode_Termination/off_side",
    "Episode_Reward/illegal_contact", "Episode_Reward/crawl", "Episode_Reward/climbed_stream",
    "Train/mean_reward", "Policy/mean_noise_std",
)


def finite(points):
    return [(p["step"], p["value"]) for p in points if isinstance(p["value"], float)]


for t in TASKS:
    art = SESSION / "artifacts" / t
    art.mkdir(parents=True, exist_ok=True)
    ARTIFACTS[t] = art
    for name in ("runtime.json",):
        shutil.copy2(SESSION / name, art / name)

    # ── Curve: tutte le scalari (JSON+CSV+PNG ufficiali) e un pannello della scalinata ──
    metrics_json = art / "metrics.json"
    run([PYTHON, "tools/colab.py", "metrics", "--run", RUN_DIR[t], "--out", metrics_json,
         "--plot", art / "training-curves.png"], cwd=REPO)
    tags = json.loads(metrics_json.read_text())["tags"]
    present = [k for k in KEY_TAGS if k in tags and finite(tags[k])]
    if present:
        cols = 3
        rows = math.ceil(len(present) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 2.8 * rows), squeeze=False)
        for ax, key in zip(axes.flat, present, strict=False):
            xs, ys = zip(*finite(tags[key]), strict=False)
            ax.plot(xs, ys, lw=1)
            ax.set_title(key, fontsize=9)
            ax.grid(alpha=0.3)
        for ax in list(axes.flat)[len(present):]:
            ax.axis("off")
        fig.suptitle(f"{t} — {RUN_DIR[t].name}")
        fig.tight_layout()
        fig.savefig(art / "stairs-curves.png", dpi=110)
        plt.close(fig)
        display(Image(filename=str(art / "stairs-curves.png")))
    summary = {}
    for key in KEY_TAGS:
        pts = finite(tags.get(key, []))
        if pts:
            last = pts[-1]
            tail = [v for _, v in pts[-20:]]
            summary[key] = {"last_step": last[0], "last": last[1], "mean_last_20": sum(tail) / len(tail)}
    (art / "stairs-summary.json").write_text(json.dumps(summary, indent=2))
    print(f"\n=== {t}: ultime 20 iterazioni (media) ===")
    for key, v in summary.items():
        print(f"  {key:<42} {v['mean_last_20']:>10.4f}   (ultimo {v['last']:.4f} @ it {v['last_step']})")

    # ── Replay MP4: una riga del curriculum, dal pianerottolo ──
    level = summary.get("Curriculum/stairs/level", {}).get("mean_last_20", 0.0)
    row = REPLAY_ROW if REPLAY_ROW != "" else str(max(0, min(9, int(round(level)))))
    rates_file = SESSION / f"rates-{t}.json"
    run([PYTHON, "tools/colab.py", "rates", "--repo", REPO, "--task", t, "--model", "jumper", "--out", rates_file], cwd=REPO)
    rates = json.loads(rates_file.read_text())
    steps = math.ceil(REPLAY_SECONDS * rates["control_hz"])
    video = art / f"replay-{t}-row{row}.mp4"
    try:
        run([PYTHON, "scripts/play.py", "--task", t, "--model", "jumper", "--checkpoint", CHECKPOINT[t],
             "--backend", "warp", "--device", "cuda:0", "--num_envs", "1", "--headless", "--steps", str(steps),
             "--physics-hz", str(rates["physics_hz"]), "--video", video, "--video-fps", str(REPLAY_FPS),
             "--video-width", str(REPLAY_WIDTH), "--video-height", str(REPLAY_HEIGHT), "--strip-visual", "off",
             "--measure", "--stairs-row", row, "--replay-info-out", art / "replay-info.json"], cwd=REPO, env=train_env(0))
        display(Video(str(video), embed=True))
        for name in ("measure.csv", "measure.png"):
            src = CHECKPOINT[t].parent / "measure" / name
            if src.is_file():
                shutil.copy2(src, art / name)
    except Exception as exc:  # noqa: BLE001 -- il replay non deve impedire il backup
        print(f"[replay {t}] fallito: {exc}")
        video = None

    # ── Export ONNX: solo la task cieca può passare il controllo di esportabilità ──
    export_dir = None
    if t == "jumper.stairs":
        export_dir = SESSION / f"onnx-{t}"
        try:
            run([PYTHON, "scripts/export.py", "--task", t, "--model", "jumper", "--checkpoint", CHECKPOINT[t],
                 "--backend", "warp", "--device", "cuda:0", "--num_envs", "1", "--no-video", "--out", export_dir],
                cwd=REPO, env=train_env(0))
        except Exception as exc:  # noqa: BLE001
            print(f"[export {t}] fallito: {exc}")
            export_dir = None
    else:
        print(f"[export {t}] saltato: questa task non è esportabile per costruzione")

    # ── Backup ZIP: checkpoint + manifest + curve + replay + export, in /kaggle/working ──
    backup = BACKUPS / f"{t}-{STAMP}.zip"
    cmd = [PYTHON, "tools/colab.py", "backup", "--run", RUN_DIR[t], "--out", backup, "--extras", art]
    if export_dir:
        cmd += ["--export", export_dir]
    if video and video.is_file():
        cmd += ["--video", video]
    run(cmd, cwd=REPO)
    print(f"BACKUP {t}: {backup} ({backup.stat().st_size / 1e6:.0f} MB)")

print("\nFATTO. Ora: Save Version → Save & Run All (Commit) conserva /kaggle/working come output.")
print("Prossima sessione: Add Input → questo notebook (versione precedente) → WORKFLOW_MODE = 'Continue training'.")


## Se qualcosa va storto

- **`nconmax overflow (nconmax must be >= N)`** nel mini-run: nel Dataset, in `tasks/jumper/stairs/env_cfg.py`, porta `cfg.sim.nconmax` esattamente a N e rilancia.
- **`cannot resume from ...`** all'inizio del run lungo: il warm start non combacia con la rete (osservazioni cambiate). Metti `WARM_START = False` per partire da zero, e segnalamelo.
- **`resume configuration differs`** in Continue: commit, `NUM_ENVS` o versioni dei pacchetti diversi dal backup. `SOURCE_REVISION` e `NUM_ENVS` devono essere quelli del backup; le versioni le reinstalla la cella 2.
- **Sessione interrotta prima della cella 5**: i checkpoint sono in `/kaggle/temp` e si perdono. Il run lungo salva ogni 50 iterazioni, ma il backup lo fa solo la cella 5: tieni `TRAIN_HOURS` ≤ 10 così resta un'ora per chiudere.
- **Il replay fallisce** ma il backup c'è: va bene, il video si rifà in un'altra sessione dal backup.
